In [15]:
import xarray as xr
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.compose import TransformedTargetRegressor



In [16]:
min_lon, min_lat, max_lon, max_lat =[-61.0, -47.375, -60.0, -44.875]
min_time, max_time = pd.to_datetime("2013-01-01"), pd.to_datetime("2023-12-31")

fishing_ds = xr.open_dataset("../data/processed/targets/cpue_HKP.nc")

fishing = fishing_ds["CPUE"]
fishing = fishing.fillna(0)


temp_ds = xr.open_dataset("../data/processed/dynamic/to_surface.nc")
temp = temp_ds["to"]
temp = (temp - temp.mean()) / temp.std()
temp = temp.fillna(0)

temp_lag1 = temp.shift(time=1).fillna(0)

temp_bottom_ds = xr.open_dataset("../data/processed/dynamic/temp_bottom.nc")
temp_bottom = temp_bottom_ds["to"]
temp_bottom = (temp_bottom - temp_bottom.mean()) / temp_bottom.std()
temp_bottom = temp_bottom.fillna(0)

temp_bottom_lag1 = temp_bottom.shift(time=1).fillna(0)

chl_ds = xr.open_dataset("../data/processed/dynamic/chl.nc")
chl = chl_ds["CHL"]
chl = (chl - chl.mean()) / chl.std()
chl = chl.fillna(0)

chl_lag1 = chl.shift(time=1).fillna(0)

mixed_ds = xr.open_dataset("../data/processed/dynamic/mixed_layer.nc")
mixed = mixed_ds["mlotst"]
mixed = (mixed - mixed.mean()) / mixed.std()
mixed = mixed.fillna(0)

mixed_lag1 = mixed.shift(time=1).fillna(0)

depth_ds = xr.open_dataset("../data/processed/static/depth.nc")
depth = depth_ds["depth"] 
depth = (depth - depth.mean()) / depth.std()
depth = depth.fillna(0)
depth = depth.broadcast_like(temp)

zo_ds = xr.open_dataset("../data/processed/dynamic/zo_surface.nc")
zo = zo_ds["zo"]
zo = (zo - zo.mean()) / zo.std()
zo = zo.fillna(0)

zo_lag1 = zo.shift(time=1).fillna(0)

so_ds = xr.open_dataset("../data/processed/dynamic/so_surface.nc")
so = so_ds["so"]
so = (so - so.mean()) / so.std()
so = so.fillna(0)

mask_ds = xr.open_dataset("../data/processed/static/fishing_area_mask.nc")
mask = mask_ds["mask"]
mask = mask.broadcast_like(temp)

month = temp["time"].dt.month
month_sin = np.sin(2 * np.pi * month / 12)
month_cos = np.cos(2 * np.pi * month / 12)
month_sin = month_sin.broadcast_like(temp)
month_cos = month_cos.broadcast_like(temp)
month = month.broadcast_like(temp)

year = temp["time"].dt.year
year = (year - year.mean()) / year.std()
year = year.broadcast_like(temp)

lat = (temp["lat"] - temp["lat"].mean()) / temp["lat"].std()
lon = (temp["lon"] - temp["lon"].mean()) / temp["lon"].std()
lat = lat.broadcast_like(temp)
lon = lon.broadcast_like(temp) #se añade como dinámica porque ya se ha corregido la forma

temp, temp_bottom, chl, mixed, fishing, mask, depth, month_sin, month_cos, lat, lon, zo, so, month, year, temp_lag1, mixed_lag1, temp_bottom_lag1, chl_lag1, zo_lag1 = xr.align(temp, temp_bottom, chl, mixed, fishing, mask, depth, month_sin, month_cos, lat, lon, zo, so, month, year, temp_lag1, mixed_lag1, temp_bottom_lag1, chl_lag1, zo_lag1, join="inner")

cropped = lambda da: da.sel(
    lon=slice(min_lon, max_lon),
    lat=slice(min_lat, max_lat),
    time=slice(min_time, max_time)
)

temp = cropped(temp)
temp_bottom = cropped(temp_bottom)
chl = cropped(chl)
mixed = cropped(mixed)
mixed_lag1 = cropped(mixed_lag1)
fishing = cropped(fishing)
mask = cropped(mask)
depth = cropped(depth)
zo = cropped(zo)
so = cropped(so)
month = cropped(month)
month_sin = cropped(month_sin)
month_cos = cropped(month_cos)
lat = cropped(lat)
lon = cropped(lon)
year = cropped(year)
temp_lag1 = cropped(temp_lag1)
temp_bottom_lag1 = cropped(temp_bottom_lag1)
chl_lag1 = cropped(chl_lag1)
zo_lag1 = cropped(zo_lag1)

In [17]:
X = xr.Dataset({
    "temp": temp,
    "temp_bottom": temp_bottom,
    "chl": chl,
    "mixed": mixed,
    "depth": depth,
    "zo": zo,
    "so": so,
    "mask": mask,
    "month": month,
    "lat": lat,
    "lon": lon,
    "year": year,
    "temp_lag1": temp_lag1,
    "temp_bottom_lag1": temp_bottom_lag1,
    "zo_lag1": zo_lag1,


})


time = X.time

train_time = time < np.datetime64("2020-01-01")  
test_time  = ~train_time

X_train_3d = X.sel(time=train_time)
X_test_3d  = X.sel(time=~train_time)


# regression target
y_catch_train_3d = fishing.sel(time=train_time)
y_catch_test_3d = fishing.sel(time=test_time)


X_train_df = X_train_3d.to_dataframe().reset_index()
X_test_df  = X_test_3d.to_dataframe().reset_index()


# catch
y_catch_train_df = y_catch_train_3d.to_dataframe(name="catch").reset_index()

y_catch_test_df = y_catch_test_3d.to_dataframe(name="catch").reset_index()

train_df = (
    X_train_df
    .merge(y_catch_train_df, on=["time", "lat", "lon"])
)

test_df = (
    X_test_df
    .merge(y_catch_test_df, on=["time", "lat", "lon"])
)


def add_spatial_features(df):
    df = df.copy()

    # spatial interactions
    df["lat_lon"] = df["lat"] * df["lon"]

    # environmental interactions
    df["temp_depth"] = df["temp"] * df["depth"]

    return df

train_df = add_spatial_features(train_df)
test_df = add_spatial_features(test_df)


y_catch_train = train_df["catch"].values
y_catch_test = test_df["catch"].values

drop_cols = ["time", "catch"]

X_train = train_df.drop(columns=drop_cols)
X_test = test_df.drop(columns=drop_cols)

In [18]:
rf = RandomForestRegressor(
    n_estimators=1000,
    max_depth=20,
    min_samples_leaf=5,
    min_samples_split=10,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_catch_train)

y_pred = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_catch_test, y_pred))
mae = mean_absolute_error(y_catch_test, y_pred)
r2 = r2_score(y_catch_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

feature_importance_df = pd.DataFrame({"feature": X_train.columns, "importance": rf.feature_importances_}).sort_values(by="importance", ascending=False)
print(feature_importance_df)

RMSE: 1.5379209667231504
MAE: 1.0259464292273683
R2: 0.5685745486344086
             feature  importance
6              depth    0.150644
1                lon    0.114937
15           lat_lon    0.099476
0                lat    0.075857
9               mask    0.067223
7                 zo    0.055049
13  temp_bottom_lag1    0.053053
14           zo_lag1    0.051326
5              mixed    0.050411
3        temp_bottom    0.049057
16        temp_depth    0.041318
2               temp    0.039185
12         temp_lag1    0.039013
4                chl    0.036391
10             month    0.031177
8                 so    0.028043
11              year    0.017839


In [19]:
gbr = HistGradientBoostingRegressor(
    max_depth=20,
    learning_rate=0.1,
    max_iter=100,
    min_samples_leaf=20,
    l2_regularization=0.1,
    random_state=42,
)

gbr.fit(X_train, y_catch_train)
y_pred = gbr.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_catch_test, y_pred))
mae = mean_absolute_error(y_catch_test, y_pred)
r2 = r2_score(y_catch_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)


RMSE: 1.5354336765939656
MAE: 0.9649822464748599
R2: 0.5699689148467466
